In [93]:
from adjusted_simobs import adjusted
from UHI_statistics import avail_thresh

In [ ]:
# Get the heat extremes of JJA for which the minimum station data availability is > 80% (the avail_thresh)
adjusted = adjusted.where(adjusted.count(dim='station') >= len(adjusted.station)*avail_thresh,drop=True)
extremes = adjusted.drop_vars(['tasmin', 'tasmax','tas','elev']).sel(time=adjusted.time.dt.season == 'JJA').mean(dim='station')
q99 = extremes.quantile(0.99).drop_vars('quantile')
extremes = extremes.where(extremes >= q99,drop=True)
print(q99)

<xarray.Dataset> Size: 48B
Dimensions:   ()
Data variables:
    tasmin_S  float64 8B 21.98
    tasmax_S  float64 8B 33.39
    tasmin_C  float64 8B 23.18
    tasmin_T  float64 8B 23.73
    tasmax_C  float64 8B 37.17
    tasmax_T  float64 8B 37.33


In [95]:
fields = ['tasmin_S', 'tasmax_S', 'tasmin_T', 'tasmax_T', 'tasmin_C', 'tasmax_C']

# Find the number of contiguous extreme heat days, 
for field in fields:
    dates = extremes.where(extremes[field].notnull(),drop=True).time.values
    gaps = np.diff(dates).astype('timedelta64[D]').astype(int) # If there's less than a 3-day gap between is part of the same event
    n_samples = (gaps >= 3).sum() + 1
    print(f'{field}: {n_samples} samples')

tasmin_S: 16 samples
tasmax_S: 13 samples
tasmin_T: 18 samples
tasmax_T: 15 samples
tasmin_C: 20 samples
tasmax_C: 14 samples
